In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration # Import the tokenizer and the text generator

# Select the t5 small model
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
# Try a longer input text
from datasets import load_dataset
ds = load_dataset("xsum", split="test[:5]")
documents = ds[:5]['document']


In [12]:
documents[4]

'Restoring the function of the organ - which helps control blood sugar levels - reversed symptoms of diabetes in animal experiments.\nThe study, published in the journal Cell, says the diet reboots the body.\nExperts said the findings were "potentially very exciting" as they could become a new treatment for the disease.\nThe experiments were on mice put on a modified form of the "fasting-mimicking diet".\nWhen people go on it they spend five days on a low calorie, low protein, low carbohydrate but high unsaturated-fat diet.\nIt resembles a vegan diet with nuts and soups, but with around 800 to 1,100 calories a day.\nThen they have 25 days eating what they want - so overall it mimics periods of feast and famine.\nPrevious research has suggested it can slow the pace of ageing.\nBut animal experiments showed the diet regenerated a special type of cell in the pancreas called a beta cell.\nThese are the cells that detect sugar in the blood and release the hormone insulin if it gets too high

In [18]:
def compare_summaries(document):
  '''
  A function that takes in text, return 4 different summaries in the shape of a dictionary
  '''
  inputs = tokenizer('summarize: ' + document, return_tensors='pt', max_length=512, truncation=True) # Returns a tensor representing the text tokens
  outputs_greedy = model.generate(inputs['input_ids'], max_length=50) # Perform greedy search (always select most likely token)
  outputs_beam = model.generate(inputs['input_ids'], max_length=50, num_beams=4) # Perform beam search (select 4 most probable candidates each step)
  outputs_low_temp = model.generate(inputs['input_ids'], max_length=50, do_sample=True, temperature=0.7) # With temperature (Changes softmax logit values)
  outputs_nucleus = model.generate(inputs['input_ids'], max_length=50, do_sample=True, top_p=0.9) # Nucleus sampling, consider most likely tokens untill their probability sum hits 0.9

  summary_greedy = tokenizer.decode(outputs_greedy[0], skip_special_tokens=True)
  summary_beam = tokenizer.decode(outputs_beam[0], skip_special_tokens=True)
  summary_low_temp = tokenizer.decode(outputs_low_temp[0], skip_special_tokens=True)
  summary_nucleus = tokenizer.decode(outputs_nucleus[0], skip_special_tokens=True)

  return {
      'Greedy' : summary_greedy,
      'Beam (n_beams=4)' : summary_beam,
      'Low Temp (T=0.7)' : summary_low_temp,
      'Nucleus (top_p=0.9)' : summary_nucleus
  }

In [20]:
all_summaries = []

In [21]:
# Loop through 5 documents, get 4 summaries for each
i = 1

for document in documents:
  summaries = compare_summaries(document)
  print(f'Summaries for Document {i}: {summaries}')
  all_summaries.append(summaries)
  i += 1

Summaries for Document 1: {'Greedy': 'prison link Cymru says some ex-offenders were living rough for up to a year. the charity says it is cheaper than jailing homeless repeat offenders. the housing act in Wales was introduced in 2015.', 'Beam (n_beams=4)': 'prison link Cymru says some ex-offenders were living rough for up to a year before finding suitable accommodation. change to the Housing Act in Wales removed the right for prison leavers to be given priority for accommodation ', 'Low Temp (T=0.7)': 'prison link cycleru says some ex-offenders were living rough for up to a year. the charity says it was cheaper to get help to address housing problems. the change to the Housing Act now applies to women.', 'Nucleus (top_p=0.9)': 'prison link Cymru says prison leavers living rough for up to a year. change in housing act in Wales removed right for prison leavers to be given priority. figures show 830 one-bedroom properties were'}
Summaries for Document 2: {'Greedy': 'police say three firea

In [23]:
all_lengths = []

In [26]:
# Compute how many tokens the output of every token is
for document_summaries in all_summaries:
  lengths = [len(tokenizer.tokenize(summary)) for summary in document_summaries.values()]
  all_lengths.append(lengths)

In [27]:
all_lengths

[[45, 47, 46, 47],
 [36, 34, 34, 35],
 [47, 47, 46, 47],
 [27, 27, 27, 47],
 [33, 47, 43, 32]]